# 2. Chopped AMBIT light: TCSPC lifetimes of Fs and Fm' inside dark gaps

**Why.** While the AMBIT actinic light or saturating pulse is on, the leaf fluorescence the LED
excites reaches the SPAD at about 40 times the laser-excited signal (runs of 3 Nov 2026). An ND
filter scales both by the same factor, so it only trades SPAD saturation (OD 2) for photon
starvation (OD 4). Here the AMBIT cuts short dark gaps into every lit line and only the SPAD
photons inside the gaps are kept: there the floor is the stray-light leak plus a little delayed
fluorescence, and the laser can run near 1 Mcps.

**How.**
1. The AMBIT firmware verb `chop,<gap_us>,<period_us>` arms the gaps for the next `arrun` only
   (branch `feat/chop-gaps` of the ambit repo). Each gap is one AS7341 LED-register write off and
   one on, placed after the ADPD read-out, so the AMBIT's own samples are unaffected.
2. The Felix time-tags every SPAD photon (TTM module, binary file). SYNC is **not** tagged:
   pylibtdc rebuilds all 80 M laser pulses per second into the file (0.7 GB per second).
3. `ambit_chop.analyse` finds the lines and the gaps in the SPAD rate, rebuilds each photon's
   laser micro-time from the photons' own periodicity (5 ms phase blocks; the laser wanders
   ~2.5 ns per second against the Felix clock), and fits Fo, Fs and Fm'.

**Needs.** AMBIT with the chop firmware (step 1 checks), laser on at 80 MHz, and enough laser
photons on the SPAD: about 50-100 kcps laser-correlated in the dark, i.e. ND about OD 2 instead
of OD 4 (step 2 checks). No cable between the AMBIT and the Felix. Close TDC-Software first.

Set `SIMULATE = True` to run every step on synthetic data (`chop_sim.py`) without hardware.

In [ ]:
import sys, time, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))      # run the notebook from scripts/ambit-gate
import importlib, ambit_gate as ag, ambit_chop as ac, chop_sim as cs
ag, ac, cs = importlib.reload(ag), importlib.reload(ac), importlib.reload(cs)

SIMULATE = False       # True: no hardware, synthetic runs through the same analysis
PORT = None            # e.g. "COM23" to skip the port scan
LASER_HZ = 80e6        # laser mode selector
THRESHOLD_V = 1.2      # SYNC and SPAD are 2.5 V LVTTL
GAP_US = 150           # requested dark time; the leaf is dark for GAP_US + ~118 us (one I2C write)
CHOP_PERIOD_US = 1000  # gap spacing: dark fraction ~ (GAP_US + 118) / CHOP_PERIOD_US, ~27 % here
RUNS = Path("runs")

## 1. AMBIT and its chop verb

The firmware answers `chop: armed gap_us=... period_us=...`. No answer means the AMBIT runs a
firmware without the verb: build and flash `feat/chop-gaps` of the ambit repo
(`pio run -t upload --upload-port COMxx`). Arming is one-shot: every run consumes it.

In [ ]:
if not SIMULATE:
    try:
        ambit.close()          # re-running must not leave the port locked
    except NameError:
        pass
    ambit = ag.find_ambit(PORT)
    print(ac.arm_chop(ambit, GAP_US, CHOP_PERIOD_US)["reply"])
    ac.disarm_chop(ambit)
else:
    print("SIMULATE: no AMBIT")

## 2. Felix and the micro-time self-test (laser on, AMBIT LED off)

Three recordings of the same light:
- the hardware histogram (SYNC-referenced): the reference decay;
- 0.3 s of SPAD + SYNC tags: the micro-times folded from the photons are compared photon by
  photon with the SYNC truth (the 0.3 s file is ~250 MB and is deleted afterwards);
- 2 s of SPAD tags only, the measurement mode.

Usable when the folded peak is as narrow as the hardware one and the photon-by-photon rms is a
few ps. If it says *not usable*, there are too few laser photons: take ND out (OD 2 or less),
check the laser is on, then repeat. On 3 Nov 2026 with OD 4: 3.3 kcps of laser on 27 kcps of
floor, not enough.

In [ ]:
if not SIMULATE:
    try:
        felix.close()
    except NameError:
        pass
    felix = ag.FelixLink().open()
    felix.set_thresholds(THRESHOLD_V)
    st = ac.selftest(felix, RUNS / "selftest", seconds=2.0, sync_seconds=0.3)
    decays = [("hardware histogram", st["hardware"]), ("time tags, folded", st.get("spad_only", {}))]
else:
    d = cs.simulate_run(RUNS / "sim_selftest", [ac.line(2.0, 10, 0)], chop=False)
    r = ac.decay_from_ttm(d / "ttm.bin", 1 / cs.T_LASER_S, clock="fold")
    print(f"simulated dark recording: folded peak FWHM {r['fwhm_ps']:.0f} ps, phase block noise "
          f"{r['phase_noise_ps']:.1f} ps, wander {r['wander_ps']:.0f} ps")
    decays = [("time tags, folded (simulated)", r)]
fig, ax = plt.subplots(figsize=(8, 3.5))
for name, r in decays:
    if r.get("ok") or "h" in r:
        h = r["h"] - r["floor_per_bin"]
        t = (np.arange(h.size) + 0.5) * r["bin_s"] * 1e9
        k = 20
        hs = np.convolve(h, np.ones(k) / k, mode="same")
        ax.semilogy(t, np.clip(hs / hs.max(), 1e-3, None), label=name)
ax.set_xlabel("micro-time [ns]"); ax.set_ylabel("normalised"); ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 3. Protocol

`phi2_protocol(X)`: Fs in a chopped actinic line at byte `X`, then the chopped saturating pulse
(255), then `X` again and a short dark line. `fvfm_protocol()`: dark Fo, chopped pulse, dark.
Lit lines sample at 10 Hz (Fs) and 100 Hz (pulse): at 100 Hz the firmware fits 9 gaps per ADPD
period, at 10 Hz 99. The table gives the gaps, the fraction of the light the leaf loses to them,
and the counting time the TCSPC gets. Light lines are chopped only if they are lit (byte > 3).

In [ ]:
MODE = "phi2"          # "phi2" or "fvfm"
ACTINIC = 0          # actinic byte (mA of the AS7341 LED driver) for Fs
if MODE == "phi2":
    PROTOCOL = ac.phi2_protocol(ACTINIC, fs_s=3.0, pulse_s=0.8, after_s=1.0, post_dark_s=0.5)
else:
    PROTOCOL = ac.fvfm_protocol(dark_s=3.0, pulse_s=0.8, post_s=1.0)
print(ag.build_arrun(PROTOCOL).strip())
print(ac.describe_plan(PROTOCOL, GAP_US, CHOP_PERIOD_US))

## 4. Pre-illumination (Phi2 only, optional)

Fs is a steady state: give the leaf minutes of the same chopped actinic light first. The LED stays
on (persist) until the measurement starts. Skipped when `PREILLUM_S = 0`.

In [ ]:
PREILLUM_S = 0         # e.g. 180
if MODE == "phi2" and PREILLUM_S > 0 and not SIMULATE:
    ac.preilluminate(ambit, ACTINIC, PREILLUM_S, GAP_US, CHOP_PERIOD_US)

## 5. Measure

Arms the gaps, starts the SPAD time tags, sends the `arrun`, stops the tags at the end of the
light and collects the AMBIT arrays and its `chopstat` (gaps executed, measured dark times, I2C
write times). The run folder holds `ttm.bin` (9 bytes per photon, ~30 MB per second of light),
`ambit.csv` and `run.json`.

In [ ]:
run_dir = RUNS / (time.strftime("%Y%m%d_%H%M%S") + f"_chop_{MODE}")
if SIMULATE:
    cs.simulate_run(run_dir, PROTOCOL, GAP_US, CHOP_PERIOD_US)
else:
    ac.run_chopped(ambit, felix, PROTOCOL, run_dir, gap_us=GAP_US, period_us=CHOP_PERIOD_US,
                   ttm_mode=dict(sync=False), threshold_v=THRESHOLD_V, laser_hz=LASER_HZ)

## 6. Analyse

The line table says, per line, whether its position was measured (LED-on interval found) and how
it was analysed: `gaps` (only photons inside the gaps: 10 us after the LED goes dark until 8 us
before the shortest dark time the firmware measured; each line's gaps are stacked and the dark
step must sit where expected, else the line is not gated) or `all photons` (gaps not found reliably, usually a low actinic
level where the LED-excited floor is small anyway; the flat floor is subtracted).

Estimators of Phi2 = 1 - Fs/Fm' (and Fv/Fm = 1 - Fo/Fm):
- `tau_shared`: amplitude-weighted lifetimes from one 2-exponential fit with lifetimes shared
  between the two decays. The most robust lifetime estimator in simulations.
- `intensity`: laser-correlated photons per second of gap, dead-time corrected. Needs constant
  laser power and geometry.
- `intensity_early`, `t_mean_early`: intensity and model-free mean arrival from the first 80 us
  of each gap only. A clear difference to the full-gap values means the yield relaxes while the
  LED is dark (see the gap profiles in step 7).
- `tau_mean`, `tau_2exp`, `tau_1exp`: model-free mean arrival and independent tail fits, kept for
  comparison; without an IRF the independent fits scatter a lot.

In simulations with the 3 Nov 2026 bench numbers (OD 2, actinic 100) `tau_shared` and `intensity`
land within a few hundredths of the true Phi2; the AMBIT chopped Fm matched the unchopped Fm on
the bench (0.1002-0.1015 against 0.1003-0.1024).
- `ambit`: the AMBIT's own s/r ratio in the same windows.

In [ ]:
res = ac.analyse(run_dir)
ac.save(res)

## 7. Plots

Top left: SPAD rate (1 ms) with the lit lines, AMBIT ratio on the right axis. Top right: 5 ms
inside the pulse at 1 us resolution, detected gaps in blue, counted part in green. Middle left:
laser-correlated rate and mean arrival time of the counted photons in 50 ms bins (yield
kinetics). Middle right: floor-subtracted decays. Bottom: inside the gap, for Fs and Fm': the
laser-correlated rate, the flat floor (delayed fluorescence decays here) and the mean arrival.

In [ ]:
fig = ac.plot(res, save_to=run_dir / "chop_summary.png", show=True)

In [ ]:
fig = ac.plot(res, save_to=run_dir / "chop_summary.png", show=True)

## 8. Series: Phi2 against the AMBIT over several actinic levels

One chopped Phi2 run per actinic byte (optionally with pre-illumination at each level), then the
lifetime Phi2 against the AMBIT Phi2. A slope near 1 and a small offset is the target.

In [ ]:
SERIES = [0, 10, 20,30,40, 50, 100]       # actinic bytes
SERIES_PREILLUM_S = 30         # pre-illumination at each level (s)
rows = []
for x in SERIES:
    prot = ac.phi2_protocol(x, fs_s=3.0, pulse_s=0.8, after_s=1.0, post_dark_s=0.5)
    d = RUNS / (time.strftime("%Y%m%d_%H%M%S") + f"_chop_phi2_x{x}")
    if SIMULATE:
        phi2_true = 0.55 * np.exp(-x / 180)                     # a made-up light response
        cs.simulate_run(d, prot, GAP_US, CHOP_PERIOD_US, leaf=cs.Leaf(tau_mp=1.2, tau_s=1.2 * (1 - phi2_true)),
                        seed=x, log=lambda *a: None)
    else:
        if SERIES_PREILLUM_S > 0:
            ac.preilluminate(ambit, x, SERIES_PREILLUM_S, GAP_US, CHOP_PERIOD_US)
        ac.run_chopped(ambit, felix, prot, d, gap_us=GAP_US, period_us=CHOP_PERIOD_US,
                       ttm_mode=dict(sync=False), threshold_v=THRESHOLD_V, laser_hz=LASER_HZ, log=lambda *a: None)
    r = ac.analyse(d, log=lambda *a: None)
    ac.save(r)
    e = r["estimates"].get("phi2", {})
    rows.append(dict(actinic=x, run=d.name, truth=(r["truth"] or {}).get("phi2"),
                     **{k: e.get(k, float("nan")) for k in ("ambit", "tau_shared", "intensity", "intensity_early")}))
    print(f"actinic {x:3d}: AMBIT {rows[-1]['ambit']:.3f}  tau_shared {rows[-1]['tau_shared']:.3f}  "
          f"intensity {rows[-1]['intensity']:.3f}" + (f"  (truth {rows[-1]['truth']:.3f})" if rows[-1]['truth'] else ""))
(RUNS / f"chop_series_{time.strftime('%Y%m%d_%H%M%S')}.json").write_text(json.dumps(rows, indent=1))
a = np.array([r_["ambit"] for r_ in rows]); b = np.array([r_["tau_shared"] for r_ in rows])
c = np.array([r_["intensity"] for r_ in rows])
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(a, b, "o", label="tau_shared"); ax.plot(a, c, "s", mfc="none", label="intensity")
lim = [0, max(0.8, np.nanmax(np.r_[a, b, c]) + 0.05)]
ax.plot(lim, lim, "k--", lw=0.8)
ok = np.isfinite(a) & np.isfinite(b)
if ok.sum() >= 2:
    k_, q_ = np.polyfit(a[ok], b[ok], 1)
    ax.set_title(f"tau_shared = {k_:.2f} x AMBIT + {q_:.3f}")
ax.set_xlabel("Phi2 AMBIT"); ax.set_ylabel("Phi2 lifetime / intensity"); ax.legend(); ax.grid(alpha=0.3)
plt.show()

In [ ]:
SERIES = [0, 10, 50, 100]       # actinic bytes
SERIES_PREILLUM_S = 0         # pre-illumination at each level (s)
rows = []
for x in SERIES:
    prot = ac.phi2_protocol(x, fs_s=3.0, pulse_s=0.8, after_s=1.0, post_dark_s=0.5)
    d = RUNS / (time.strftime("%Y%m%d_%H%M%S") + f"_chop_phi2_x{x}")
    if SIMULATE:
        phi2_true = 0.55 * np.exp(-x / 180)                     # a made-up light response
        cs.simulate_run(d, prot, GAP_US, CHOP_PERIOD_US, leaf=cs.Leaf(tau_mp=1.2, tau_s=1.2 * (1 - phi2_true)),
                        seed=x, log=lambda *a: None)
    else:
        if SERIES_PREILLUM_S > 0:
            ac.preilluminate(ambit, x, SERIES_PREILLUM_S, GAP_US, CHOP_PERIOD_US)
        ac.run_chopped(ambit, felix, prot, d, gap_us=GAP_US, period_us=CHOP_PERIOD_US,
                       ttm_mode=dict(sync=False), threshold_v=THRESHOLD_V, laser_hz=LASER_HZ, log=lambda *a: None)
    r = ac.analyse(d, log=lambda *a: None)
    ac.save(r)
    e = r["estimates"].get("phi2", {})
    rows.append(dict(actinic=x, run=d.name, truth=(r["truth"] or {}).get("phi2"),
                     **{k: e.get(k, float("nan")) for k in ("ambit", "tau_shared", "intensity", "intensity_early")}))
    print(f"actinic {x:3d}: AMBIT {rows[-1]['ambit']:.3f}  tau_shared {rows[-1]['tau_shared']:.3f}  "
          f"intensity {rows[-1]['intensity']:.3f}" + (f"  (truth {rows[-1]['truth']:.3f})" if rows[-1]['truth'] else ""))
(RUNS / f"chop_series_{time.strftime('%Y%m%d_%H%M%S')}.json").write_text(json.dumps(rows, indent=1))
a = np.array([r_["ambit"] for r_ in rows]); b = np.array([r_["tau_shared"] for r_ in rows])
c = np.array([r_["intensity"] for r_ in rows])
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(a, b, "o", label="tau_shared"); ax.plot(a, c, "s", mfc="none", label="intensity")
lim = [0, max(0.8, np.nanmax(np.r_[a, b, c]) + 0.05)]
ax.plot(lim, lim, "k--", lw=0.8)
ok = np.isfinite(a) & np.isfinite(b)
if ok.sum() >= 2:
    k_, q_ = np.polyfit(a[ok], b[ok], 1)
    ax.set_title(f"tau_shared = {k_:.2f} x AMBIT + {q_:.3f}")
ax.set_xlabel("Phi2 AMBIT"); ax.set_ylabel("Phi2 lifetime / intensity"); ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 9. Re-analyse a saved run

Any run folder can be analysed again, e.g. with other windows (`fs_last_s`, `fm_skip_s`), guards
or `early_us`.

In [ ]:
# res = ac.analyse(RUNS / "20261103_160000_chop_phi2", fs_last_s=1.0, fm_skip_s=0.25, early_us=80)
# fig = ac.plot(res, show=True)

## 10. Close both instruments

In [ ]:
if not SIMULATE:
    felix.close()
    ambit.close()